# Feature Engineering — Non-Temporal Approach

This notebook transforms event-level smartphone sensing data into an **instance-based tabular dataset**
suitable for standard (non-temporal) ML models.

Each row is one prediction instance: *patient i on date t*, using only information from
the `window_size` calendar days **before** `t`. The target is the mean mood on day `t`.

### Pipeline overview
1. **Daily aggregation** — collapse event records to one row per (patient, calendar day)
2. **Window feature engineering** — for each target day, aggregate the preceding window
3. **Build & save** — produce train/val/test parquet files for every candidate window size

The downstream classification notebook (`2a_classification_non_temporal/classification.ipynb`)
handles all model training, hyperparameter search, and evaluation.

In [25]:
import sys
import warnings
from pathlib import Path
from typing import Dict, List
import os

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from scipy import stats  # noqa: E402

from src.utils.data import get_windows  # noqa: E402

warnings.filterwarnings("ignore")

In [26]:
DATA_DIR = Path(os.getcwd()).parent.parent / "data"
SOURCE_DATA_FILE_CLEANED = DATA_DIR / "1b_dataset_cleaned.parquet"

df = pd.read_parquet(SOURCE_DATA_FILE_CLEANED)
print(
    f"Loaded {len(df):,} records | "
    f"{df['id'].nunique()} patients | "
    f"{df['variable'].nunique()} variables"
)
print(f"Date range: {df['time'].min().date()} to {df['time'].max().date()}")
df.head()

Loaded 376,413 records | 27 patients | 19 variables
Date range: 2014-02-17 to 2014-06-09


,id,variable,time,value
0,AS14.01,activity,2014-03-20 22:00:00,0.071429
1,AS14.01,activity,2014-03-20 23:00:00,0.091667
2,AS14.01,activity,2014-03-21 00:00:00,0.008333
3,AS14.01,activity,2014-03-21 01:00:00,0.000000
4,AS14.01,activity,2014-03-21 02:00:00,0.000000


In [27]:
# ============================================================
# PIPELINE CONFIGURATION
# ============================================================

# Candidate window sizes to compare (calendar days of history before the target day).
WINDOW_SIZES: List[int] = [1, 3, 5, 7, 14]
DEFAULT_WINDOW: int = 7

OUT_DIR = Path(os.getcwd()) / "output"
OUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# Variable grouping — determines the daily aggregation method.
# ------------------------------------------------------------------

# MEAN_VARS: continuous ratings or activity rates.
MEAN_VARS: List[str] = [
    "mood",
    "circumplex.arousal",
    "circumplex.valence",
    "activity",
]

# SUM_VARS: durations (seconds) or binary event flags.
SUM_VARS: List[str] = [
    "screen",
    "call",
    "sms",
    "appCat.builtin",
    "appCat.communication",
    "appCat.entertainment",
    "appCat.finance",
    "appCat.game",
    "appCat.office",
    "appCat.other",
    "appCat.social",
    "appCat.travel",
    "appCat.unknown",
    "appCat.utilities",
    "appCat.weather",
]

APP_CAT_VARS: List[str] = [v for v in SUM_VARS if v.startswith("appCat.")]

known_vars = set(df["variable"].unique())
assert not (set(MEAN_VARS) - known_vars), (
    f"Unknown MEAN_VARS: {set(MEAN_VARS) - known_vars}"
)
assert not (set(SUM_VARS) - known_vars), (
    f"Unknown SUM_VARS: {set(SUM_VARS) - known_vars}"
)

print(
    f"Configuration OK | "
    f"{len(MEAN_VARS)} mean-vars, {len(SUM_VARS)} sum-vars | "
    f"Window sizes to compare: {WINDOW_SIZES}"
)

Configuration OK | 4 mean-vars, 15 sum-vars | Window sizes to compare: [1, 3, 5, 7, 14]


---
## Step 1: Daily Aggregation

Collapse event-level records to one row per **(patient, calendar day)**.

### Aggregation rules

| Variable group | Method | Rationale |
|---|---|---|
| `mood`, `circumplex.arousal/valence`, `activity` | **mean** | Ratings/rates; average of multiple daily readings gives the daily level |
| `screen`, `appCat.*` | **sum** | Session durations; total daily usage is the meaningful quantity |
| `call`, `sms` | **sum** | Values are always 1.0 in this dataset → sum = event count |

### Missingness preservation

For every variable we also produce:
- `{var}_n_obs` — number of raw event records on that day
- `{var}_observed` — binary flag (1 = at least one valid observation that day)

Days with **no records at all** for a variable remain as `NaN` in the primary column
and `0` in the `_observed` column. This keeps the signal of absence (e.g., no screen
usage on a day is different from low screen usage).

In [28]:
def aggregate_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate event-level long-form data to patient-day wide format.

    Parameters
    ----------
    df : pd.DataFrame
        Long-form DataFrame with columns [id, variable, time, value].

    Returns
    -------
    pd.DataFrame
        Wide-form with one row per (id, date). Columns:
          id, date,
          {var}_mean   (MEAN_VARS),
          {var}_sum    (SUM_VARS),
          {var}_n_obs  (all variables),
          {var}_observed (binary, all variables).
        Variable names are sanitised: dots replaced by underscores.
    """
    df = df.copy()
    # Floor timestamps to midnight so all readings on the same calendar day
    # belong to the same group regardless of hour.
    df["date"] = df["time"].dt.normalize()

    grouped = df.groupby(["id", "date", "variable"])["value"]
    agg_mean = grouped.mean()  # used for MEAN_VARS
    agg_sum = grouped.sum()  # used for SUM_VARS
    agg_count = grouped.count()  # used for all: counts non-NaN values

    var_in_data = set(agg_count.index.get_level_values("variable"))

    frames: List[pd.Series] = []

    for var_orig in MEAN_VARS:
        if var_orig not in var_in_data:
            continue
        vs = var_orig.replace(".", "_")  # safe column name
        frames.append(agg_mean.xs(var_orig, level="variable").rename(f"{vs}_mean"))
        frames.append(agg_count.xs(var_orig, level="variable").rename(f"{vs}_n_obs"))

    for var_orig in SUM_VARS:
        if var_orig not in var_in_data:
            continue
        vs = var_orig.replace(".", "_")
        frames.append(agg_sum.xs(var_orig, level="variable").rename(f"{vs}_sum"))
        frames.append(agg_count.xs(var_orig, level="variable").rename(f"{vs}_n_obs"))

    # Combine: rows = all observed (id, date) combos; missing combos for a
    # given variable become NaN (outer join behaviour of pd.concat axis=1).
    daily_wide = pd.concat(frames, axis=1)
    daily_wide.index.names = ["id", "date"]
    daily_wide = daily_wide.reset_index()
    daily_wide.columns.name = None

    # Binary observed indicators
    for var_orig in MEAN_VARS + SUM_VARS:
        vs = var_orig.replace(".", "_")
        nobs_col = f"{vs}_n_obs"
        if nobs_col in daily_wide.columns:
            # fillna(0) so that NaN (never in data) maps to 0, not True
            daily_wide[f"{vs}_observed"] = (daily_wide[nobs_col].fillna(0) > 0).astype(
                np.int8
            )

    return daily_wide.sort_values(["id", "date"]).reset_index(drop=True)

In [29]:
daily_wide = aggregate_to_daily(df)

n_patients = daily_wide["id"].nunique()
n_patient_days = len(daily_wide)
mood_obs_per_patient = daily_wide.groupby("id")["mood_observed"].sum()

print("Daily aggregation complete.")
print(f"  Shape            : {daily_wide.shape}")
print(f"  Patient-day rows : {n_patient_days:,}")
print(f"  Patients         : {n_patients}")
print(
    f"  Date range       : {daily_wide['date'].min().date()} to {daily_wide['date'].max().date()}"
)
print(
    f"  Mood obs / patient (min/median/max): "
    f"{mood_obs_per_patient.min():.0f} / "
    f"{mood_obs_per_patient.median():.0f} / "
    f"{mood_obs_per_patient.max():.0f}"
)

print("\nColumn overview (first 12):")
print(daily_wide.columns[:12].tolist())
daily_wide.head(3)

Daily aggregation complete.
  Shape            : (1973, 59)
  Patient-day rows : 1,973
  Patients         : 27
  Date range       : 2014-02-17 to 2014-06-09
  Mood obs / patient (min/median/max): 30 / 46 / 68

Column overview (first 12):
['id', 'date', 'mood_mean', 'mood_n_obs', 'circumplex_arousal_mean', 'circumplex_arousal_n_obs', 'circumplex_valence_mean', 'circumplex_valence_n_obs', 'activity_mean', 'activity_n_obs', 'screen_sum', 'screen_n_obs']


,id,date,mood_mean,mood_n_obs,circumplex_arousal_mean,circumplex_arousal_n_obs,circumplex_valence_mean,circumplex_valence_n_obs,activity_mean,activity_n_obs,...,appCat_entertainment_observed,appCat_finance_observed,appCat_game_observed,appCat_office_observed,appCat_other_observed,appCat_social_observed,appCat_travel_observed,appCat_unknown_observed,appCat_utilities_observed,appCat_weather_observed
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [30]:
# Build a per-patient daily lookup dict used for lag-based feature lookups in
# _compute_window_features. Each entry is the patient's full daily wide DataFrame
# indexed by date (includes all dates, NaN rows for days without observations).
daily_wide_by_patient: Dict[str, pd.DataFrame] = {}
for pid, pdf in daily_wide.groupby("id"):
    pat = pdf.drop(columns="id").sort_values("date").set_index("date")
    full_range = pd.date_range(pat.index.min(), pat.index.max(), freq="D")
    daily_wide_by_patient[pid] = pat.reindex(full_range)

print(f"Patient daily lookup ready: {len(daily_wide_by_patient)} patients")

Patient daily lookup ready: 27 patients


---
## Steps 2 & 3: Target Construction and Window Feature Engineering

### Target
For each **(patient, target_date)** instance:
- `target_mood` = mean of all mood readings on `target_date`
- Only days where mood was observed become target dates.

### Leakage prevention (by construction)
The history window for target date `t` spans **`[t - window_size, t - 1]`** inclusive.
No data from day `t` is ever included in features.
Target dates without at least `window_size` days of history are dropped.

### Feature families

| Family | Variables | Notes |
|---|---|---|
| Rolling mean / std / min / max | mood, arousal, valence, activity | Typical level and variability over window |
| Rolling sum / mean / max | screen, call, sms, appCat.* | Behavioral volume |
| Lag features | mood: full set up to `w`; others: lag-1 only | Recency; mood yesterday is very predictive |
| Same-weekday lag | mood | Weekly periodicity |
| Linear trend (slope) | mood (≥3 observed days) | Direction of change |
| Missingness indicators | all variables | Absence of signal may be informative |
| App category proportions | appCat.* | Usage composition, independent of volume |
| Temporal context | target date | Cyclical day-of-week, weekend flag |
| Window coverage | all | How much of the window had any data |

In [31]:
def _compute_window_features(
    window: pd.DataFrame,
    patient_daily: pd.DataFrame,
    target_date: pd.Timestamp,
    window_size: int,
) -> Dict:
    """
    Compute all features for a single (patient, target_date) instance.

    Parameters
    ----------
    window : pd.DataFrame
        Daily wide data for the window [target_date-window_size, target_date-1].
        Indexed by Timestamp. NaN rows = calendar days with no observations.
        Length is always exactly window_size (guaranteed by caller).
    patient_daily : pd.DataFrame
        Full patient data indexed by date. Used for lag lookups that may fall
        outside the current window (e.g. same-weekday lag).
    target_date : pd.Timestamp
        Prediction target day. NOT included in window.
    window_size : int
        Number of calendar days in the window.
    """
    feats: Dict = {}

    # ------------------------------------------------------------------
    # Temporal context of the TARGET day
    # ------------------------------------------------------------------
    dow = target_date.dayofweek  # 0 = Monday, 6 = Sunday
    feats["target_dow"] = dow
    feats["target_is_weekend"] = int(dow >= 5)
    feats["target_dow_sin"] = float(np.sin(2 * np.pi * dow / 7))
    feats["target_dow_cos"] = float(np.cos(2 * np.pi * dow / 7))

    # ------------------------------------------------------------------
    # Overall window coverage
    # ------------------------------------------------------------------
    any_obs = window.notna().any(axis=1)
    feats["window_n_any_obs_days"] = int(any_obs.sum())
    feats["window_coverage_frac"] = float(any_obs.mean())

    # ------------------------------------------------------------------
    # MEAN variables: mood, circumplex.arousal/valence, activity
    # ------------------------------------------------------------------
    for var_orig in MEAN_VARS:
        vs = var_orig.replace(".", "_")
        mean_col = f"{vs}_mean"
        obs_col = f"{vs}_observed"

        if mean_col not in window.columns:
            continue

        series = window[mean_col]
        obs = (
            window[obs_col].fillna(0)
            if obs_col in window.columns
            else series.notna().astype(float)
        )
        n_obs = int(obs.sum())

        feats[f"{vs}_win_mean"] = float(series.mean()) if n_obs > 0 else np.nan
        feats[f"{vs}_win_std"] = float(series.std()) if n_obs > 1 else np.nan
        feats[f"{vs}_win_min"] = float(series.min()) if n_obs > 0 else np.nan
        feats[f"{vs}_win_max"] = float(series.max()) if n_obs > 0 else np.nan

        feats[f"{vs}_n_obs_days"] = n_obs
        feats[f"{vs}_obs_frac"] = n_obs / window_size
        feats[f"{vs}_n_missing_days"] = window_size - n_obs

        max_lag = window_size if var_orig == "mood" else 1
        for lag in range(1, max_lag + 1):
            lag_date = target_date - pd.Timedelta(days=lag)
            if lag_date in patient_daily.index and mean_col in patient_daily.columns:
                feats[f"{vs}_lag{lag}"] = patient_daily.at[lag_date, mean_col]
            else:
                feats[f"{vs}_lag{lag}"] = np.nan

        swd_date = target_date - pd.Timedelta(days=7)
        if swd_date in patient_daily.index and mean_col in patient_daily.columns:
            feats[f"{vs}_same_wday_lag7"] = patient_daily.at[swd_date, mean_col]
        else:
            feats[f"{vs}_same_wday_lag7"] = np.nan

        if var_orig == "mood":
            valid = series.dropna()
            if len(valid) >= 3:
                x = np.arange(len(valid), dtype=float)
                slope, *_ = stats.linregress(x, valid.values)
                feats[f"{vs}_trend"] = float(slope)
            else:
                feats[f"{vs}_trend"] = np.nan

    # ------------------------------------------------------------------
    # SUM variables: screen, call, sms, appCat.*
    # ------------------------------------------------------------------
    for var_orig in SUM_VARS:
        vs = var_orig.replace(".", "_")
        sum_col = f"{vs}_sum"
        obs_col = f"{vs}_observed"

        if sum_col not in window.columns:
            continue

        obs = (
            window[obs_col].fillna(0)
            if obs_col in window.columns
            else window[sum_col].notna().astype(float)
        )
        n_obs = int(obs.sum())

        series_filled = window[sum_col].fillna(0)
        series_raw = window[sum_col]

        feats[f"{vs}_win_sum"] = float(series_filled.sum())
        feats[f"{vs}_win_mean"] = float(series_filled.mean())
        feats[f"{vs}_win_max"] = float(series_raw.max()) if n_obs > 0 else np.nan
        feats[f"{vs}_n_obs_days"] = n_obs
        feats[f"{vs}_obs_frac"] = n_obs / window_size

    # ------------------------------------------------------------------
    # App category proportions
    # ------------------------------------------------------------------
    total_app = sum(
        window[f"{v.replace('.', '_')}_sum"].fillna(0).sum()
        for v in APP_CAT_VARS
        if f"{v.replace('.', '_')}_sum" in window.columns
    )
    for var_orig in APP_CAT_VARS:
        vs = var_orig.replace(".", "_")
        sum_col = f"{vs}_sum"
        if sum_col in window.columns:
            var_total = window[sum_col].fillna(0).sum()
            feats[f"{vs}_prop"] = (var_total / total_app) if total_app > 0 else np.nan
        else:
            feats[f"{vs}_prop"] = np.nan

    return feats


def _aggregate_window_events(
    window_events: pd.DataFrame,
    window_size: int,
    target_date: pd.Timestamp,
) -> pd.DataFrame:
    """
    Convert tall-format window events (output of get_windows) to the daily wide
    format expected by _compute_window_features. The result is indexed by date and
    covers exactly `window_size` calendar days ending the day before `target_date`;
    days with no events appear as NaN rows.
    """
    frames: List[pd.Series] = []
    present_vars = set(window_events["variable"].unique())

    for var_orig in MEAN_VARS:
        if var_orig not in present_vars:
            continue
        vs = var_orig.replace(".", "_")
        grp = window_events.loc[window_events["variable"] == var_orig].groupby("date")[
            "value"
        ]
        frames.append(grp.mean().rename(f"{vs}_mean"))
        frames.append(grp.count().rename(f"{vs}_n_obs"))

    for var_orig in SUM_VARS:
        if var_orig not in present_vars:
            continue
        vs = var_orig.replace(".", "_")
        grp = window_events.loc[window_events["variable"] == var_orig].groupby("date")[
            "value"
        ]
        frames.append(grp.sum().rename(f"{vs}_sum"))
        frames.append(grp.count().rename(f"{vs}_n_obs"))

    daily = pd.concat(frames, axis=1) if frames else pd.DataFrame()

    # Reindex to cover every calendar day in the window (empty days → NaN)
    window_dates = pd.date_range(
        target_date - pd.Timedelta(days=window_size),
        target_date - pd.Timedelta(days=1),
        freq="D",
    )
    daily = daily.reindex(window_dates)
    daily.index.name = "date"

    for var_orig in MEAN_VARS + SUM_VARS:
        vs = var_orig.replace(".", "_")
        nobs_col = f"{vs}_n_obs"
        if nobs_col in daily.columns:
            daily[f"{vs}_observed"] = (daily[nobs_col].fillna(0) > 0).astype(np.int8)

    return daily


def build_feature_matrix_v2(window_size: int) -> pd.DataFrame:
    """
    Build an instance-based feature matrix using the standardised user-based splits.

    Uses get_windows() to obtain windowed raw events together with their pre-assigned
    train/val/test split labels (70/10/20 % of patients from
    1b_dataset_target_splits.parquet). Each window is aggregated to daily format via
    _aggregate_window_events and then passed to _compute_window_features.

    Parameters
    ----------
    window_size : int
        Number of calendar days of history per instance (>= 1).

    Returns
    -------
    pd.DataFrame
        One row per (id, target_date). First columns: id, target_date, target_mood,
        split. Remaining columns: engineered features.
    """
    raw = get_windows(window_size=window_size)

    records = []
    for window_id, group in raw.groupby("window_id"):
        patient_id = group["id"].iloc[0]
        target_date = pd.Timestamp(group["date_target"].iloc[0])
        target_mood = float(group["target_mean_mood"].iloc[0])
        split = group["split"].iloc[0]

        patient_daily = daily_wide_by_patient.get(patient_id)
        if patient_daily is None:
            continue

        window_daily = _aggregate_window_events(group, window_size, target_date)
        feats = _compute_window_features(
            window=window_daily,
            patient_daily=patient_daily,
            target_date=target_date,
            window_size=window_size,
        )
        feats["id"] = patient_id
        feats["target_date"] = target_date
        feats["target_mood"] = target_mood
        feats["split"] = split
        records.append(feats)

    result = pd.DataFrame(records)
    meta = ["id", "target_date", "target_mood", "split"]
    feat_cols = [c for c in result.columns if c not in meta]
    return result[meta + feat_cols].reset_index(drop=True)

In [32]:
print(f"Building feature matrices for window sizes: {WINDOW_SIZES} ...")
_feature_dfs: Dict[int, pd.DataFrame] = {}

for ws in WINDOW_SIZES:
    feat_df = build_feature_matrix_v2(ws)
    _feature_dfs[ws] = feat_df

    meta_cols_all = ["id", "target_date", "target_mood", "split"]
    fc = [c for c in feat_df.columns if c not in meta_cols_all]
    print(f"\nw={ws:>2}d  instances={len(feat_df):>4}  features={len(fc)}")

    for split_name in ["train", "val", "test"]:
        split_df = feat_df[feat_df["split"] == split_name].drop(columns="split")
        out_path = OUT_DIR / f"{split_name}_w{ws}.parquet"
        split_df.to_parquet(out_path, index=False)
        print(f"  Saved {out_path.name}  ({len(split_df)} rows)")

    feat_df.to_parquet(OUT_DIR / f"features_w{ws}.parquet", index=False)

# Expose default-window frame + feat_cols for the inspection cell below
feature_df = _feature_dfs[DEFAULT_WINDOW]
meta_cols = ["id", "target_date", "target_mood", "split"]
feat_cols = [c for c in feature_df.columns if c not in meta_cols]

print(f"\nAll window sizes saved to {OUT_DIR}/")

Building feature matrices for window sizes: [1, 3, 5, 7, 14] ...

w= 1d  instances=1259  features=130
  Saved train_w1.parquet  (858 rows)
  Saved val_w1.parquet  (136 rows)
  Saved test_w1.parquet  (265 rows)

w= 3d  instances=1268  features=132
  Saved train_w3.parquet  (864 rows)
  Saved val_w3.parquet  (136 rows)
  Saved test_w3.parquet  (268 rows)

w= 5d  instances=1268  features=134
  Saved train_w5.parquet  (864 rows)
  Saved val_w5.parquet  (136 rows)
  Saved test_w5.parquet  (268 rows)

w= 7d  instances=1268  features=136
  Saved train_w7.parquet  (864 rows)
  Saved val_w7.parquet  (136 rows)
  Saved test_w7.parquet  (268 rows)

w=14d  instances=1268  features=143
  Saved train_w14.parquet  (864 rows)
  Saved val_w14.parquet  (136 rows)
  Saved test_w14.parquet  (268 rows)

All window sizes saved to /Users/jortverbeek/Desktop/VU/data mining/Assignment 1/data_mining_group_9_assignment_1/src/1c_feature_engineering_non_temporal/output/


In [34]:
# Feature missingness breakdown by family
families = {
    "mood_lags": [c for c in feat_cols if "mood_lag" in c],
    "mood_rolling": [c for c in feat_cols if c.startswith("mood_win")],
    "mood_misc": [
        c
        for c in feat_cols
        if c.startswith("mood_") and "lag" not in c and "win" not in c
    ],
    "arousal": [c for c in feat_cols if "arousal" in c],
    "valence": [c for c in feat_cols if "valence" in c],
    "activity": [c for c in feat_cols if c.startswith("activity_")],
    "screen": [c for c in feat_cols if "screen" in c],
    "call_sms": [c for c in feat_cols if "call" in c or "sms" in c],
    "appCat": [c for c in feat_cols if "appCat" in c and "prop" not in c],
    "appCat_prop": [c for c in feat_cols if "appCat" in c and "prop" in c],
    "temporal": [c for c in feat_cols if "target_" in c],
    "window_cov": [c for c in feat_cols if c.startswith("window_")],
}

summary_rows = []
for family, cols in families.items():
    if cols:
        miss = feature_df[cols].isna().mean().mean()
        summary_rows.append(
            {"family": family, "n_features": len(cols), "missingness": f"{miss:.1%}"}
        )

pd.DataFrame(summary_rows).set_index("family")

,n_features,missingness
family,,
mood_lags,7,10.5%
mood_rolling,4,2.9%
mood_misc,4,3.4%
arousal,9,4.4%
valence,9,4.4%
activity,9,10.1%
screen,5,5.4%
call_sms,10,7.6%
appCat,60,36.2%


---
## Step 4: Feature Inspection (default window = 7 days)

The cell above saved parquets for all window sizes. Below we inspect the missingness
profile of the default window (`w=7`) to verify feature quality before passing the data
to the classification notebook.

---
## Summary of design decisions

### Output
For each window size in `WINDOW_SIZES`, three parquet files are written to `output/`:
`train_w{ws}.parquet`, `val_w{ws}.parquet`, `test_w{ws}.parquet`.
The classification notebook loads these and decides which window size and model perform best.

### Data split
All feature matrices use the **standardised user-based splits** from
`1b_dataset_target_splits.parquet` (via `get_windows()`):
70 % train / 10 % val / 20 % test, allocated by patient.
This is identical to the split used by the RNN notebook, enabling fair cross-model comparison.

### Aggregation
- `mood`, `arousal`, `valence`, `activity` → **daily mean**
- `screen`, `appCat.*` → **daily sum** (session durations)
- `call`, `sms` → **daily sum** (values are always 1.0 → sum = event count)

### Missingness
- Days with no records contribute `NaN` to the primary column and `0` to `_observed`.
- No imputation is applied here; models in the classification notebook handle it.

### Feature design
- **Full lag set for mood** (lag₁ … lagₙ where n = window_size); other MEAN_VARS: lag₁ only.
- **Same-weekday lag (lag₇)** captures weekly rhythms without requiring a 7-day window.
- **Linear slope** of mood over the window (≥ 3 observed points).
- **App proportion** features normalise by total app time (composition vs volume).
- **Cyclical day-of-week encoding** (sin/cos).